# Review aliases — stats, evidence, approve / reject
One scope only. Read-only stats first, then explicit per-alias or bulk review. Every write goes through `decide_alias()` — the same guarded path as `review-alias` CLI — so audit rules hold.


In [1]:
from pathlib import Path
import json
from sqlalchemy import text
from potency_kg.db import create_engine_from_env
from potency_kg.workflow_stages import (
    inspect_aliases_stage, inspect_alias_evidence_stage,
    review_alias_stage, alias_status_stage,
)
from potency_kg.aliases import AliasDecision

nb_dir = Path.cwd().resolve()
package_root = nb_dir.parent if nb_dir.name == "notebooks" else nb_dir
config_path = package_root / "configs/casgevy-potency.json"
scope = json.loads(config_path.read_text())["scope"]
TENANT, PROJECT = scope["tenant_id"], scope["project_id"]
engine = create_engine_from_env()
PARAMS = {"tenant": TENANT, "project": PROJECT}
print(f"scope: {TENANT}/{PROJECT}")
print(f"config: {config_path}")


scope: public-fda/casgevy-potency
config: /workspace/configs/casgevy-potency.json


## Where aliases live in the DB
Postgres 17 (`db:5432` in compose, `localhost:5432` from host; connect via `POSTGRES_*` env, see `lib/potency_kg/db.py`). Schema: `migrations/versions/0005_create_scoped_alias_registry.py`. All rows scoped by `(tenant_id, project_id)`.
- `aliases` — one row per `(normalized_alias, entity_type, canonical_key, canonical_name)` candidate. Key columns: `alias_id` (scoped hash of that tuple), `alias_value` (verbatim), `normalized_alias` (NFKC+casefold), `entity_type / canonical_key / canonical_name` (NULL until reviewed), `status` (`pending|approved|rejected`), `managed_by` (`static_config|review`), `decided_by/at/note`, `confidence`. PK `(tenant,project,alias_id)`; the only dedup guarantee is partial index `uq_aliases_one_approved_target ON (tenant,project,entity_type,normalized_alias) WHERE status='approved'` — `pending` duplicates are expected.
- `alias_evidence` — immutable occurrence rows (`exact_quote`, `original_start/end`, `passage_id`, `detector_name/version`, `input_sha256`), `source_type`: `static_config` / `deterministic_rule` / `llm_mined` / `extraction`. Joins `aliases` on `(tenant,project,alias_id)`.
- `alias_discovery_run` — per-document detector runs (`method='deterministic'|'llm'`, `status`, `counts`), for rerun-reuse auditing.
Only `approved` rows enter `approved_alias_catalog()` and get stamped as `alias_catalog_version` on `jev_response` / `llm_extraction_run`.


In [2]:
def alias_stats():
    with engine.connect() as c:
        print("== aliases.status ==")
        for s, n in c.execute(text("SELECT status, count(*) FROM aliases WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1 ORDER BY 2 DESC"), PARAMS).all():
            print(f"  {s:10s} {n}")
        print("== aliases.managed_by ==")
        for m, n in c.execute(text("SELECT managed_by, count(*) FROM aliases WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1"), PARAMS).all():
            print(f"  {m:14s} {n}")
        print("== aliases.entity_type ==")
        for e, n in c.execute(text("SELECT coalesce(entity_type,'<NULL>'), count(*) FROM aliases WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1"), PARAMS).all():
            print(f"  {e:10s} {n}")
        print("== top duplicated normalized_alias (same synonym, several rows) ==")
        for norm, n in c.execute(text("SELECT normalized_alias, count(*) FROM aliases WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1 HAVING count(*)>1 ORDER BY 2 DESC LIMIT 15"), PARAMS).all():
            print(f"  {norm!r:40s} {n} rows")
        print("== canonical_name variations per normalized_alias (top 10) ==")
        for norm, variants in c.execute(text("SELECT normalized_alias, count(DISTINCT coalesce(canonical_name,'<NULL>')) FROM aliases WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1 HAVING count(DISTINCT coalesce(canonical_name,'<NULL>'))>1 ORDER BY 2 DESC LIMIT 10"), PARAMS).all():
            print(f"  {norm!r}: {variants} distinct canonical_name")
        print("== alias_evidence.source_type ==")
        for s, n in c.execute(text("SELECT source_type, count(*) FROM alias_evidence WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1"), PARAMS).all():
            print(f"  {s:20s} {n}")
        print("== alias_discovery_run (method, status, n) ==")
        for m, s, n in c.execute(text("SELECT method, status, count(*) FROM alias_discovery_run WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1,2"), PARAMS).all():
            print(f"  {m:14s} {s:10s} {n}")

alias_stats()
print()
print("catalog:", alias_status_stage(config_path))


== aliases.status ==
  pending    330
  approved   3
== aliases.managed_by ==
  static_config  3
  review         330
== aliases.entity_type ==
  Product    3
  <NULL>     330
== top duplicated normalized_alias (same synonym, several rows) ==
  'study'                                  8 rows
  'hhspcs'                                 7 rows
  'pmr'                                    7 rows
  'eop'                                    6 rows
  'scd'                                    5 rows
  'hb'                                     5 rows
  'pmrs'                                   4 rows
  'vai'                                    4 rows
  'iv'                                     4 rows
  'ds'                                     3 rows
  'isa'                                    3 rows
  'pmc'                                    3 rows
  'nai'                                    3 rows
  'exa-cel'                                3 rows
  'moa'                                    3 rows
== cano

In [3]:
# Review queue: pending rows + their competing canonical_name variants + evidence count.
def show_queue(limit=30):
    rows = inspect_aliases_stage(config_path, status="pending")
    print(f"pending: {len(rows)}")
    for r in rows[:limit]:
        print(f"  {r.alias_id} | alias={r.alias_value!r} norm={r.normalized_alias!r} "
              f"target=({r.entity_type},{r.canonical_key},{r.canonical_name!r}) conf={r.confidence}")
    return rows

def show_variants(normalized, limit=20):
    """All rows (any status) sharing one normalized_alias — answers 'why several rows?'."""
    with engine.connect() as c:
        rows = c.execute(text("SELECT alias_id, alias_value, entity_type, canonical_key, canonical_name, status, managed_by "
                             "FROM aliases WHERE tenant_id=:tenant AND project_id=:project AND normalized_alias=:n ORDER BY status, alias_id"),
                         {**PARAMS, "n": normalized}).mappings().all()
    for r in rows[:limit]:
        print(dict(r))
    return rows

def show_evidence(alias_id):
    ev = inspect_alias_evidence_stage(config_path, alias_id=alias_id)
    print(f"evidence for {alias_id}: {len(ev)}")
    for e in ev[:5]:
        print(f"  [{e.source_type}/{e.detector_name}] passage={e.passage_id} span=({e.original_start},{e.original_end}) quote={ (e.exact_quote or '')[:160]!r}")
    return ev

queue = show_queue()


pending: 330
  alias:48fd4d8e8df17685390196eecc7f07b4 | alias='AC003' norm='ac003' target=(None,None,'assay control') conf=0.8
  alias:15f30be0803186de2cd0b25d2b56ad09 | alias='ACS' norm='acs' target=(None,None,'acute chest syndrome') conf=0.8
  alias:e53ecafdc890cfc89054d62e9acd9174 | alias='Act' norm='act' target=(None,None,'PHS Act') conf=0.8
  alias:c8c07eebe6a0cbdc586a3c84b87b10cf | alias='ActRIIA' norm='actriia' target=(None,None,'A receptor IIA') conf=0.8
  alias:ee8e07372307f4202001415a4881d407 | alias='AEs' norm='aes' target=(None,None,'adverse events') conf=0.8
  alias:ce5d9bea2b8e69ea907a30ae5cc979bf | alias='AESI' norm='aesi' target=(None,None,'Adverse Events of Special Interest') conf=0.8
  alias:a572130af15cfe412f19e4dcf64d3b12 | alias='AESIs' norm='aesis' target=(None,None,'adverse events of special interest') conf=0.8
  alias:0b4277ee93aedc18bfb6bc017d861914 | alias='AHU' norm='ahu' target=(None,None,'air handling unit') conf=0.8
  alias:4df850c75b7ecc755b4b0eafa9448994

## Review semantics (read before writing)
`review_alias_stage(config, AliasDecision(...))` → `decide_alias()` (`lib/potency_kg/aliases.py:425`): single-row `UPDATE` of `entity_type, canonical_key, canonical_name, status, decided_at/by/note`. Evidence rows are never touched. Static-seed rows (`managed_by='static_config'`) are refused.
- `pending|rejected → approved` requires `entity_type, canonical_key, canonical_name`.- `pending|approved → rejected` needs only a reviewer (+ optional note).- Identical repeat decision = no-op. `approved → approved` with a *different* target is rejected — go via `rejected` first.
**No `back-to-pending` transition exists by design** (audit trail; `rejected` rows stay re-approvable and keep collecting evidence). In this notebook `send_back_to_review()` therefore means: `approved → rejected` (removes it from the approved catalog) or confirm an already-`rejected` row is in the review pool. A decision immediately changes the approved-catalog version — explicitly rerun Jev → extraction → materialize → refresh → export afterwards.


In [4]:
REVIEWER = "notebook-reviewer"  # ponytail: single knob, set once and reuse below

def review_alias(alias_id, decision, *, entity_type=None, canonical_key=None, canonical_name=None, note=None, reviewer=REVIEWER):
    """Approve or reject one alias by alias_id. Returns the updated AliasRecord."""
    return review_alias_stage(config_path, AliasDecision(alias_id, decision, reviewer, entity_type, canonical_key, canonical_name, note))

def send_back_to_review(alias_id, *, note="sent back to review", reviewer=REVIEWER):
    """Closest thing to 'back to review': approved/pending -> rejected (stays re-approvable). Rejected -> no-op with notice."""
    current = [r for r in inspect_aliases_stage(config_path) if r.alias_id == alias_id]
    if not current:
        raise ValueError(f"unknown alias_id in this scope: {alias_id}")
    if current[0].status == "rejected":
        print(f"{alias_id} already rejected (in review pool, re-approvable). No write.")
        return current[0]
    return review_alias(alias_id, "rejected", note=note, reviewer=reviewer)

def review_all(decision, *, status_filter="pending", entity_type=None, canonical_key=None,
              canonical_name=None, note=None, reviewer=REVIEWER, dry_run=True, limit=None, alias_ids=None):
    """Bulk review. Same target applied to every selected row; per-row errors captured, never aborts the batch.
    approve-all needs explicit entity_type/canonical_key/canonical_name (deterministic proposals carry NULLs).
    Set dry_run=False to actually write."""
    rows = inspect_aliases_stage(config_path, status=status_filter) if alias_ids is None else \
        [r for r in inspect_aliases_stage(config_path) if r.alias_id in set(alias_ids)]
    if limit is not None:
        rows = rows[:limit]
    print(f"{'DRY-RUN ' if dry_run else ''}{decision} x {len(rows)} (filter={status_filter!r})")
    ok, skipped, errors = [], [], []
    for r in rows:
        if r.managed_by == "static_config":
            skipped.append((r.alias_id, "static_config"))
            continue
        if dry_run:
            ok.append(r.alias_id)
            continue
        try:
            review_alias(r.alias_id, decision, entity_type=entity_type, canonical_key=canonical_key,
                         canonical_name=canonical_name, note=note, reviewer=reviewer)
            ok.append(r.alias_id)
        except Exception as e:  # ponytail: per-row capture so one conflict never aborts the batch
            errors.append((r.alias_id, f"{type(e).__name__}: {e}"))
    print(f"ok={len(ok)} skipped_static={len(skipped)} errors={len(errors)}")
    for aid, err in errors[:10]:
        print(f"  ERROR {aid}: {err}")
    return {"ok": ok, "skipped": skipped, "errors": errors}

print("helpers ready: review_alias(), send_back_to_review(), review_all(), show_queue(), show_variants(), show_evidence()")


helpers ready: review_alias(), send_back_to_review(), review_all(), show_queue(), show_variants(), show_evidence()


## Single-alias review (edit + run)
Pick an `alias_id` from `show_queue()`, inspect its `show_evidence(alias_id)` and `show_variants(normalized)`, then run one of the three lines.


In [5]:
# Inspect one candidate first (replace with a real id from the queue above):
# show_evidence(queue[0].alias_id) if queue else print("queue empty")
# show_variants(queue[0].normalized_alias) if queue else print("queue empty")

# Then exactly one of:
# review_alias(queue[0].alias_id, "approved", entity_type="Product", canonical_key="casgevy", canonical_name="CASGEVY", note="grounded quote supports synonym")
# review_alias(queue[0].alias_id, "rejected", note="not a synonym / out of scope")
# send_back_to_review(queue[0].alias_id, note="needs second look")
print("edit the commented lines above with your alias_id, then run")


edit the commented lines above with your alias_id, then run


## Bulk review (edit + run, `dry_run=False` writes)
`for all` = same filter + same decision. Reject-all is target-free; approve-all needs one explicit target applied to every row (use `alias_ids=[...]` for a chosen subset when targets differ).


In [6]:
# 1) Preview: what would be rejected?
# review_all("rejected", status_filter="pending", dry_run=True)

# 2) Reject all pending (writes):
# review_all("rejected", status_filter="pending", note="bulk reject: not synonyms", dry_run=False)

# 3) Approve all pending under ONE target (writes; fails per-row on approved-conflicts, reported not raised):
# review_all("approved", status_filter="pending", entity_type="Product", canonical_key="casgevy", canonical_name="CASGEVY", note="bulk approve", dry_run=False)

# 4) Subset by id (mixed targets -> one call per target group):
# review_all("approved", alias_ids=["<id-1>", "<id-2>"], entity_type="Assay", canonical_key="<key>", canonical_name="<name>", dry_run=False)

# 5) Send all approved back to review (approved -> rejected, catalog shrinks):
# review_all("rejected", status_filter="approved", note="re-review requested", dry_run=False)
print("edit + set dry_run=False to write")


edit + set dry_run=False to write


## Verify after review
Re-run stats: pending should shrink, catalog version changes. Any Jev/extraction stamped with the old `alias_catalog_version` is stale — rerun `classify_jev_stage → extract_stage → materialize_stage → refresh_graph_stage(mode='semantic') → export_stage` explicitly.


In [7]:
alias_stats()
print()
print("catalog:", alias_status_stage(config_path))


== aliases.status ==
  pending    330
  approved   3
== aliases.managed_by ==
  static_config  3
  review         330
== aliases.entity_type ==
  Product    3
  <NULL>     330
== top duplicated normalized_alias (same synonym, several rows) ==
  'study'                                  8 rows
  'hhspcs'                                 7 rows
  'pmr'                                    7 rows
  'eop'                                    6 rows
  'scd'                                    5 rows
  'hb'                                     5 rows
  'pmrs'                                   4 rows
  'vai'                                    4 rows
  'iv'                                     4 rows
  'ds'                                     3 rows
  'isa'                                    3 rows
  'pmc'                                    3 rows
  'nai'                                    3 rows
  'exa-cel'                                3 rows
  'moa'                                    3 rows
== cano